day03

0. 복습
데이터 전처리
 : 분석하기 전에 데이터를 분석 목적에 맞게 변형하는 것

결측치 처리
1) 결측치 삭제
 : NaN이 있는 행 또는 열 삭제
- 결측치가 많이 없는 경우
- 한 열이 거의 다 결측치인 경우

2) 결측치 채우기
 : 결측치를 적당한 값으로 채우는 방법
- 숫자 열 : 중앙값, 평균값
- 문자열 : 최빈값

1. 이상치(outlier) 처리
1) 이상치
 : 다른 값들과 동떨어지게 유난히 크거나 작은값
- 이상치는 평균을 왜곡할 수 있다
- 머신러닝 모델 학습을 방해한다
ex) 반 학생 30명의 키가 150 ~ 180cm인데
    한명만 210cm => 이 한명이 반 평균 키를 실제보다 높여버린다
cf) 이상치가 무조건 잘못된 값인 건 아니다
    진짜로 특별한 값일 수 있다(ex : 1등석 초고가 요금)

2) 이상치 확인
(1) 시각화로 확인 : 박스그래프(boxplot)
- 박스그래프(boxplot) : 데이터가 어디에 몰려 있고, 
		어디부터가 이상치인지 한눈에 확인하는 그래프
- 박스 그래프 구성 요소
	상자(box) : 가운데 50%(2사분위)의 값이 들어 있는 범위
		    (아래 = 하위 25%지점, 1사분위
		     위 = 상위 25%지점, 3사분위)
	상자 안의 선 : 중앙값(2사분위)
	수염(whisker) : 상자에서 위아래로 뻗은 선
			(정상으로 보는 범위의 끝)
	수염 밖의 점 : 이상치로 의심되는 값

(2) 통계적인 방법 - IQR 방법 
 : 정확히 어디부터 이상치인지 기준이 필요하다
   가장 널리 사용하는 방법이 IQR 방법
- IQR(사분위 범위) : Q3(3사분위수) - Q1(1사분위수)
			(박스그래프의 박스의 높이)
- 이상치 경계 : 아래 공식으로 정상 범위를 정하고, 이 범위를 벗어나면
		이상치로 본다
	아래 경계 = Q1 - 1.5 x IQR
	위 경계 = Q3 + 1.5 x IQR
	(박스 그래프의 수염이 이 기준으로 만들어진다)

3) 이상치 처리하기
 : 이상치를 찾았다면, 상황에 따라 3가지 중 하나를 고른다

(1) 제거 : 이상치 행을 빼기
- 결측치의 dropna()처럼, 필터링(조건)으로 정상 범위 안의 
  행만 남긴다

(2) 대체 : 경계값으로 눌러담기(clip)
- 데이터를 버리기 아까울때, 이상치를 경계값까지만 끌어내려
  눌러담는다
- df['열'].clip(lower=해당 값보다 작으면 이 값으로 올림,
		upper=해당 값보다 크면 이 값으로 내림)

(3) 유지 - 그냥 두기
 : 이상치가 오류가 아니라 진짜 의미가 있는 값이라면,
   함부로 지우면 안된다.

2. 구간 분할(binning)
 : 연속된 숫자 데이터를 몇 개의 구간으로 나눠 범주(그룹)로 바꾸는 것
ex) 나이(22, 25, ...)처럼 값이 이어지는 숫자를,
    "어린이, 청소녀느 성인,..." 같은 구간으로 나누는 것
- 구간분할을 하는 이유
	데이터를 그룹으로 비교할때 사용(연령대벼르 요금 등급별 비교)
	머신러닝에서 범주형 특성으로 활용할 수 있다

1) pd.cut() : 값의 크기로 나누기
- 경계를 직접 정해서 그 구간대로 나눈다
- 옵션
	bins : 구간을 가르는 경계값 목록	[0, 18, 35, 60, 100]
	labels : 각 구간에 붙일 이름	["어린이", "청년", "중년", "노년"]
	right : 경계값을 오른쪽 구간에 포함할지
		(기본값 True)

2) pd.qcut() : 개수로 나누기(분위수)
- qcut은 "구간마다 개수를 똑같이" 나눈다
- 각 구간에 같은 수의 데이터가 들어가도록 경계를 자동으로 잡는다
- 옵션 
	q : 몇 등분할지
	labels : 각 구간 이름

In [ ]:
## 이상치 처리
import pandas as pd
import seaborn as sns

# 타이타닉 데이터 불러오기
titanic = sns.load_dataset("titanic")
# 요금 열의 통계 요약 정보
titanic["fare"].describe()
# 중앙값은 약 14.5인데 최대값은 무려 512이다 
# 몇몇이 극단적으로 비쌀 수 있다 => 이런 값이 이상치일 가능성이 크다
### 이상치 확인
import matplotlib.pyplot as plt
# 박스 그래프로 확인

plt.rcParams["font.family"] = "Malgun Gothic" # 한글 깨짐 방지

plt.figure(figsize=(6, 4))
plt.boxplot(titanic["fare"])
plt.title("요금 boxplot")
plt.ylabel("요금")
plt.show()
# 요금은 큰 쪽으로 이상치가 몰려있다
### IQR 사분위 범위
# IQR = Q3(3사분위) - Q1(1사분위)

# 사분위 수 구하기
q1 = titanic["fare"].quantile(0.25) # 1사분위 수
q3 = titanic["fare"].quantile(0.75) # 3사분위 수
# IQR 계산
iqr = q3 - q1

# 정상 범위의 아래/위 경계 계산
# 1) 위쪽 경계 : Q3 + 1.5 x IQR (이 값보다 크다면 이상치)
high = q3 + 1.5 * iqr
# 2) 아래쪽 경계 : Q1 - 1.5 x IQR (이 값보다 작다면 이상치)
low = q1 - 1.5 * iqr

print(f"Q1 : {q1 : .2f}, Q3 : {q3 : .2f}, IQR : {iqr : .2f}")
print(f"정상 범위 : {low : .2f} ~ {high : .2f}")
# 요금이 대략 65.63보다 크면 이상치로 본다 
# 정상 범위를 벗어난 행만 가져오기(이상치 가져오기)
# 아래 경계보다 작거나, 위 경계보다 크다면 이상치
outlier = titanic[(titanic['fare'] < low) | (titanic['fare'] > high)]

print(f"이상치 개수 : {len(outlier)}개")
print(f"전체 대비 : {len(outlier) / len(titanic) * 100 : .2f}%")
# 이상치들만 모아, 요금이 큰 순서로 보기
print(outlier['fare'].sort_values(ascending=False).head())
outlier.head()
# 116명(약 13%)의 요금이 이상치에 해당한다
# 512처럼 일반 요금 (중앙값 약 14.5)과 자릴수부터 다른 값들이다
# 이런 값들이 평균을 끌어올리는 이상치이다
### 이상치 처리
# 이상치 제거 : 정상범위 안의 행만 남기기
# * 정상범위 Q1 - 1.5 x IQR(하한값) ~ Q3 + 1.5 x IQR(상한값)
clean = titanic[(titanic['fare'] >= low) & (titanic['fare'] <= high)]

print(f"처리 전 : {len(titanic)}개")
print(f"처리 후 : {len(clean)}개")
# 116개 행이 빠지고 775개 행이 남음
# 제거하는 방법은 간단하지만, 데이터가 줄어든다는 게 단점이다
# 위 경계(상한값, 65.63)보다 큰 요금은 모두 65.63으로 눌러담는다
titanic['fare_clip'] = titanic['fare'].clip(lower=low, upper=high)

print(f"원래 최대값 : {titanic['fare'].max()}")
print(f"clip후 최대값 : {titanic['fare_clip'].max()}")
# clip후의 최대 요금은 경계값 65.63으로 눌렀다
# 행 수는 그대로라 데이터를 잃지 않는다
# +) 또 다른 대체법
# 이상치만 대표값으로 바꾸기 
# : 이상치를 중앙값이나 평균 같은 대표값으로 바꿀 수 있다

titanic['fare_fix'] = titanic['fare'] # fare요금 데이터 복사
# 1) 요금의 중앙값 계산
fare_median = titanic['fare'].median()
print(f"중앙값 : {fare_median : .2f}")

# 2) 요금이 위 경계(high)보다 큰 행의 칸만 중앙값으로 대체
# df.loc[조건, 열] = 값 => 조건에 맞는 행의 열 값만 바꿀 수 있다
titanic.loc[titanic['fare'] > high, "fare_fix"] = fare_median

# 3) 이상치였던 행을 골라 확인
titanic.loc[[1, 27, 31], ['fare', 'fare_fix']]
# 이상치 요금이 모두 중앙값으로 바뀜
## 구간 분할(binning)
import pandas as pd
import seaborn as sns

# 타이타닉 데이터 불러오기
titanic = sns.load_dataset("titanic")
titanic.head()
### pd.cut()
# 나이를 연령대 4개로 나눠서 구간 분할
age_group = pd.cut(
    titanic["age"], # 나눌 대상
    bins = [0, 18, 35, 60, 120], # 구간 경계
    labels= ["어린이", "청년", "중년", "노년"] # 구간이름
)
# print(age_group)

# 원래 나이와 나눈 구간확인
titanic["연령대"] = age_group
titanic[['age', '연령대']].head(6)
# 나이가 NaN이던 5번 행은 구간도 NaN이다.
# 35.0이 '청년'에 들어간 이유 : right=True(기본값)
# 경계값 35는 왼쪽 구간(18~35)에 포함된다
# => 경계값의 반대쪽으로 넣고 싶다면 => right=False를 주면된다
# right=False : 경계값을 다음 구간(오른쪽)에 포함한다.
age_group2 = pd.cut(
    titanic["age"], # 나눌 대상
    bins = [0, 18, 35, 60, 120], # 구간 경계
    labels= ["어린이", "청년", "중년", "노년"], # 구간이름
    right= False 
)
titanic["연령대2"] = age_group2
titanic[['age', '연령대', '연령대2']].head()
# 35세가 오른족 구간인 "중년"으로 바뀌어 있음
# 각 연령대에 몇명이 있는지 확인
# .sort_index() => 구간 순서대로 정렬됨
print(titanic['연령대'].value_counts().sort_index())
# 타이타닉 탑승객은 청년(18~35)이 가장 많다는 걸 한눈에 알 수 있다
# +) 경계를 일일이 정하기 힘들다면, bins=구간개수
# 그러면 최소값 ~ 최대값을 같은 폭으로 자동 분할한다
age_auto = pd.cut(titanic['age'], bins=4)
# 나이를 동일한 폭으로 4구간으로 자동 분할
print(age_auto.value_counts().sort_index())
### pd.qcut()
# 요금을 "인원수가 같도록" 4등급으로 나누기
fare_grade = pd.qcut(titanic['fare'], q = 4)
print(fare_grade.value_counts().sort_index())
# 네 구간의 인원이 거의 같다
# 구간에 이름을 붙이고 나누기
fare_grade = pd.qcut(titanic['fare'], q=4, 
                     labels=["하", "중하", "중상", "상"])
titanic["요금등급"] = fare_grade
print(titanic["요금등급"].value_counts().sort_index())
# +) 구간 활용하기 -> 그룹별로 비교하기
# groupby("열") : 그 열의 값이 같은 행끼리 묶어, 그룹별로 계산해준다

# 위에서 나눈 연령대별로 생존률 계산
print(titanic.groupby("연령대", observed=True)["survived"].mean().round(2))
# 어린이의 생존률이 가장 높고, 노년이 가장 낮다
# age(나이)를 그냥 두고 계산했다면 이런 비교가 어려운데
# 구간으로 묶으면 한번에 계산할 수 있다
# 요금 등급별 생존률
print(titanic.groupby("요금등급", observed=True)['survived'].mean().round(2))
# 요금이 높을수록 생존률이 높다
# 이렇게 연속된 숫자를 구간으로 묶으면, 그룹별 경향을 비교 및 요약할 수 있다
# => 구간 분할을 하는 이유 중 하나
# <구간 분할 실습>
# titanic 데이터 사용
# 1) age를 pd.cut()으로 10대 단위 연령대로 나누기
# - 경계 : 0, 10, 20, 30, ...
# 2) 나눈 연령대를 value_counts()로, 가장 인원이 많은 연령대가 어디인지 확인
# 3) fare을 pd.qcut()으로 3등급(저가, 중가, 고가)로 나누기
import pandas as pd
import seaborn as sns

titanic = sns.load_dataset("titanic")

# 1) 나이를 10대 단위로 나누기
titanic["연령대"] = pd.cut(
    titanic["age"],
    bins=[0, 10, 20, 30, 40, 50, 60, 80],
    labels= ["10대미만", "10대", "20대", "30대", "40대", "50대", "60대이상"]
)

# 2) 연령대별 인원수 확인(구간 순으로 정렬)
print(titanic["연령대"].value_counts().sort_index())
# 20대가 제일 많다
# 3) 요금을 3등급으로 나누기
titanic["요금등급"] = pd.qcut(titanic['fare'], q=3,
                         labels=["저가", "중가", "고가"])

# 요금 등급별 인원수 확인
print(titanic["요금등급"].value_counts().sort_index())
# 완전히 같지 않은건, 같은 요금이 어렷이라 경계에서 딱 떨어지지 않기 때문에

과제

## day03 과제 

seaborn의 **`tips`**(식당 팁 기록, 244행) 데이터로 연습합니다. 아래 셀을 먼저 실행해 `tips` 를 불러오세요.

- `total_bill` : 총 결제 금액(달러) / `tip` : 팁(달러) / `size` : 일행 인원 수 등
- 이번 과제는 `total_bill`(결제액)의 **이상치**를 다루고, `total_bill`·`tip`을 **구간으로 분할**합니다.

> 💡 `sns.load_dataset("tips")` 로 데이터 불러오기.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# 이번 과제 공통 데이터 (이 셀을 먼저 실행!)
tips = sns.load_dataset("tips")
print(tips.head())

문제 1) 이상치 눈으로 찾기 — describe와 boxplot

- `total_bill`(결제액)의 요약 통계를 `describe()`로 출력하세요.
- `total_bill`의 **상자수염그림(boxplot)**을 그려, 수염 밖에 이상치 점이 어느 쪽에 있는지 눈으로 확인하세요.
  (제목 `"total_bill boxplot"`, `figsize=(6, 4)`)

<출력결과>

count    244.000000
mean      19.785943
std        8.902412
min        3.070000
25%       13.347500
50%       17.795000
75%       24.127500
max       50.810000
Name: total_bill, dtype: float64

In [ ]:
print(tips["total_bill"].describe())

plt.figure(figsize=(6, 4))
plt.boxplot(tips["total_bill"])
plt.title("total_bill boxplot")
plt.show()

문제 2) IQR 방법으로 이상치 기준 정하기

- `total_bill`의 **Q1(하위 25%)**, **Q3(상위 25%)**, **IQR(=Q3-Q1)** 을 구하세요. (힌트 : `quantile(0.25)`, `quantile(0.75)`)
- 정상 범위의 **아래 경계(`Q1 - 1.5*IQR`)** 와 **위 경계(`Q3 + 1.5*IQR`)** 를 계산해 출력하세요.
- 정상 범위를 벗어난 **이상치가 몇 개**이고 **전체의 몇 %**인지 출력하세요. (힌트 : `(값 < low) | (값 > high)`)

<출력결과>

Q1: 13.35, Q3: 24.13, IQR: 10.78
정상 범위: -2.82 ~ 40.30
이상치 개수: 9
전체 대비: 3.7%

In [ ]:
q1 = tips["total_bill"].quantile(0.25)
q3 = tips["total_bill"].quantile(0.75)
iqr = q3 - q1
low = q1 - 1.5 * iqr
high = q3 + 1.5 * iqr

outliers = tips[(tips["total_bill"] < low) | (tips["total_bill"] > high)]

print(f"Q1 : {q1 : .2f}, Q3 : {q3 : .2f}, IQR : {iqr : .2f}")
print(f"정상 범위 : {low : .2f} ! {high : .2f}")
print(f"이상치 개수 : {len(outliers)}")
print(f"전체 대비 : {len(outliers) / len(tips) * 100 : .1f}")

문제 3) 이상치 처리 — clip(대체)과 제거 비교

- 문제 2에서 구한 `low`, `high` 를 이용합니다.
- ① **대체** : `clip(lower=low, upper=high)` 로 이상치를 경계값까지 눌러담아 새 열 `bill_clip` 에 저장하고, **처리 전후 최댓값**을 비교 출력하세요.
- ② **제거** : 정상 범위 **안**의 행만 남겨(`>= low` 그리고 `<= high`), **처리 전/후 행 수**를 출력하세요.

<출력결과>

원래 최댓값: 50.81
clip 후 최댓값: 40.29749999999999
처리 전: 244 행
처리 후: 235 행

In [ ]:
tips["bill_clip"] = tips["total_bill"].clip(lower=low, upper=high)
print(f"원래 최댓값 : {tips["total_bill"].max()}")
print(f"clip 후 최댓값 : {tips["bill_clip"].max()}")

tips_clean = tips[(tips["total_bill"] >= low) & (tips["total_bill"] <= high)]
print(f"처리 전 : {len(tips)}행")
print(f"처리 후 : {len(tips_clean)}행")

문제 4) pd.cut — 값의 크기로 결제액 등급 나누기

- `total_bill`(결제액)을 경계값 `[0, 15, 30, 60]` 으로 나눠 `"저가", "중가", "고가"` 세 등급을 붙이고, 새 열 `결제액등급` 에 저장하세요. (힌트 : `pd.cut`, `bins`, `labels`)
- 각 등급에 몇 명이 있는지 `value_counts().sort_index()` 로 세어 출력하세요.

<출력결과>

결제액등급
저가     80
중가    132
고가     32
Name: count, dtype: int64

In [ ]:
tips = sns.load_dataset("tips")

tips["결제액등급"] = pd.cut(
    tips["total_bill"],
    bins=[0, 15, 30, 60],
    labels=["저가", "중가", "고가"]
)
print(tips["결제액등급"].value_counts().sort_index())

문제 5) pd.qcut — 개수로 팁 등급 나누기

- `tip`(팁)을 **인원수가 비슷하도록 3등급**으로 나눠 `"하", "중", "상"` 이름을 붙이고, 새 열 `팁등급` 에 저장하세요. (힌트 : `pd.qcut`, `q=3`, `labels`)
- 각 등급 인원을 `value_counts().sort_index()` 로 세어, **세 등급 인원이 비슷한지** 확인하세요.

<출력결과>

팁등급
하    83
중    80
상    81
Name: count, dtype: int64

In [ ]:
tips["팁등급"] = pd.qcut(
    tips["tip"],
    q=3,
    labels=["하", "중", "상"]
)
print(tips["팁등급"].value_counts().sort_index())

문제 6) 만든 구간 활용 — 그룹별 비교

- 문제 4에서 만든 `결제액등급` 으로 묶어(`groupby`), **등급별 평균 팁(`tip`)** 을 소수 셋째 자리까지 출력하세요. (힌트 : `groupby("결제액등급", observed=True)["tip"].mean().round(3)`)
- 결제액 등급이 올라갈수록 평균 팁이 커지는지 확인하세요.

<출력결과>

결제액등급
저가    2.050
중가    3.189
고가    4.583
Name: tip, dtype: float64

In [ ]:
print(tips.groupby("결제액등급", observed=True)["tip"].mean().round(3))